# Data Investigation Notebook
Focuses on deep, recursive inspection of the exact data structures and metadata loaded by the MuSE-Toolbox data pipeline.

In [11]:
import os
import sys
from pathlib import Path
import torch
import numpy as np
from hydra import initialize, compose
from hydra.utils import instantiate
from IPython.display import display, HTML, clear_output, Audio

import logging
import sys
# Force all INFO-level logs to print directly to the notebook's standard output
logging.basicConfig(stream=sys.stdout, level=logging.INFO, force=True)

# Setup project root
PROJECT_ROOT = Path(os.getcwd()).parent
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# --- CONFIGURATION VARIABLES ---
DATASET = "pra_anf_circ_8ch"
SPLIT = "train"  # 'train', 'val', or 'test'
SCENARIO_IDX = 0

with initialize(version_base="1.3", config_path="../configs"):
    cfg = compose(config_name="default", overrides=[f"dataset={DATASET}"])
    
print(f"Instantiating datamodule for dataset: {DATASET}")
datamodule = instantiate(cfg.dataset)

base_root = datamodule.precomputed_dir

# Find all directories for this split
all_split_dirs = []
raw_dir = base_root / SPLIT
if raw_dir.exists():
    all_split_dirs.append(raw_dir)

stft_base = base_root / 'stft'
if stft_base.exists():
    all_split_dirs.extend(stft_base.glob(f"*/{SPLIT}"))

features_base = base_root / 'features'
if features_base.exists():
    all_split_dirs.extend(features_base.glob(f"*/{SPLIT}"))

if not all_split_dirs:
    raise FileNotFoundError(f"\n{'='*50}\nNo data found for split '{SPLIT}' in {base_root}!\n{'='*50}")

from muse_toolbox.data.components.precomputed_dataset import PrecomputedDataset
datasets = []
for typepath in all_split_dirs:
    datasets.append(PrecomputedDataset(precomputed_dir=typepath, preload_to_ram=False))
 

print(f"Dataset length (total files across all directories): {len(datasets[0])}")

# Determine the target scenario ID from SCENARIO_IDX
print(f"Extracting all data for Scenario ID: {SCENARIO_IDX}")

scenario_data = {"raw": None, "stft": [], "features": []}

for dataset in datasets:
    item = dataset[SCENARIO_IDX]
    input_type = item.get('input_type')
    if input_type == 'raw_audio':
        scenario_data['raw'] = item
    elif input_type == 'stft':
        # Remove the 'meta' key and its contents
        item.pop('meta', None)
        scenario_data['stft'].append(item)
    elif input_type == 'features':
        # Remove the 'meta' key and its contents
        item.pop('meta', None)
        scenario_data['features'].append(item)

item = scenario_data # assign to item so the next cell can print it
print(f"Successfully loaded all data for scenario {SCENARIO_IDX}")
print(f"Found: {1 if scenario_data['raw'] is not None else 0} raw, {len(scenario_data['stft'])} STFTs, {len(scenario_data['features'])} features")


Instantiating datamodule for dataset: pra_anf_circ_8ch
INFO:muse_toolbox.data.datamodules.PRA_ANF:Found implementation for clean speech database: LibrispeechDatabase
INFO:muse_toolbox.data.databases.librispeech:LibrispeechDatabase initialized to use splits: ['train-clean-360', 'dev-clean', 'test-clean']
INFO:muse_toolbox.data.datamodules.PRA_ANF:Found implementation for noise source database: DemandDatabase
INFO:muse_toolbox.data.databases.demand:DemandDatabase will download the 16kHz version.
INFO:muse_toolbox.data.databases.demand:DemandDatabase initialized for 16kHz.
INFO:muse_toolbox.data.databases.demand:  - train split will use 12 environments.
INFO:muse_toolbox.data.databases.demand:  - val split will use 3 environments.
INFO:muse_toolbox.data.databases.demand:  - test split will use 3 environments.
INFO:muse_toolbox.data.datamodules.PRA_ANF:Initialized PraRIRsDB for on-the-fly RIR generation.
INFO:muse_toolbox.data.datamodules.PRA_ANF:Initialized AnfNoiseDB to use a noise DB fo

In [12]:
from muse_toolbox.utils.debug_utils import print_structure

print("=== Recursive Data Structure Investigation ===")
print_structure(item)


=== Recursive Data Structure Investigation ===
INFO:muse_toolbox.utils.debug_utils:dict with 3 keys | Memory: 156.02 MB
INFO:muse_toolbox.utils.debug_utils:  key: 'raw'
INFO:muse_toolbox.utils.debug_utils:    dict with 3 keys | Memory: 90.34 MB
INFO:muse_toolbox.utils.debug_utils:      key: 'meta'
INFO:muse_toolbox.utils.debug_utils:        dict with 8 keys | Memory: 75.69 MB
INFO:muse_toolbox.utils.debug_utils:          key: 'scenario_params'
INFO:muse_toolbox.utils.debug_utils:            dict with 15 keys | Memory: 48.45 KB
INFO:muse_toolbox.utils.debug_utils:              key: 'num_sources'
INFO:muse_toolbox.utils.debug_utils:                int | Memory: 28 B
INFO:muse_toolbox.utils.debug_utils:              key: 'signal_length'
INFO:muse_toolbox.utils.debug_utils:                float | Memory: 24 B
INFO:muse_toolbox.utils.debug_utils:              key: 'snr'
INFO:muse_toolbox.utils.debug_utils:                float | Memory: 24 B
INFO:muse_toolbox.utils.debug_utils:             

In [14]:
8000 * 60

480000